# Notebook interactif : Opérations OLAP en SQL

Ce notebook vous permet de **pratiquer** les opérations OLAP directement dans votre navigateur, sans avoir besoin d'installer PostgreSQL ou BigQuery.

On utilise **DuckDB**, une base de données analytique légère qui supporte toutes les fonctions SQL OLAP (ROLLUP, CUBE, GROUPING SETS, Window Functions).

**Comment utiliser ce notebook :**
1. Exécutez les cellules dans l'ordre (Shift + Enter)
2. Lisez les explications avant chaque requête
3. Modifiez les requêtes pour expérimenter
4. Les exercices marqués ✏️ sont à compléter par vous

In [ ]:
# Installation de DuckDB (une seule fois)
!pip install duckdb -q

In [ ]:
import duckdb

# Connexion DuckDB en mémoire (rien à configurer !)
con = duckdb.connect()

def sql(query):
    """Exécute une requête SQL et affiche le résultat sous forme de DataFrame pandas."""
    return con.sql(query).df()

print("DuckDB prêt !")

---
## 1. Création du mini Data Warehouse

On crée un jeu de données de ventes réaliste : une chaîne de magasins avec des ventes par région, catégorie et trimestre.

In [ ]:
# Création des tables
con.sql("""
CREATE TABLE fact_ventes AS
SELECT * FROM (VALUES
    (2024, 'T1', 'Île-de-France', 'Paris',     'Électronique', 'Smartphones', 120000),
    (2024, 'T1', 'Île-de-France', 'Paris',     'Électronique', 'Laptops',      95000),
    (2024, 'T1', 'Île-de-France', 'Paris',     'Vêtements',    'Hauts',         80000),
    (2024, 'T1', 'Île-de-France', 'Paris',     'Vêtements',    'Chaussures',    45000),
    (2024, 'T1', 'Île-de-France', 'Versailles','Électronique', 'Smartphones',   35000),
    (2024, 'T1', 'Auvergne-RA',   'Lyon',      'Électronique', 'Smartphones',   45000),
    (2024, 'T1', 'Auvergne-RA',   'Lyon',      'Vêtements',    'Hauts',         30000),
    (2024, 'T1', 'PACA',          'Marseille',  'Électronique', 'Laptops',       40000),
    (2024, 'T1', 'PACA',          'Marseille',  'Vêtements',    'Chaussures',    25000),
    (2024, 'T2', 'Île-de-France', 'Paris',     'Électronique', 'Smartphones', 150000),
    (2024, 'T2', 'Île-de-France', 'Paris',     'Électronique', 'Laptops',     110000),
    (2024, 'T2', 'Île-de-France', 'Paris',     'Vêtements',    'Hauts',         95000),
    (2024, 'T2', 'Île-de-France', 'Paris',     'Vêtements',    'Chaussures',    55000),
    (2024, 'T2', 'Île-de-France', 'Versailles','Électronique', 'Smartphones',   42000),
    (2024, 'T2', 'Auvergne-RA',   'Lyon',      'Électronique', 'Smartphones',   55000),
    (2024, 'T2', 'Auvergne-RA',   'Lyon',      'Vêtements',    'Hauts',         35000),
    (2024, 'T2', 'PACA',          'Marseille',  'Électronique', 'Laptops',       48000),
    (2024, 'T2', 'PACA',          'Marseille',  'Vêtements',    'Chaussures',    28000),
    (2024, 'T3', 'Île-de-France', 'Paris',     'Électronique', 'Smartphones',  90000),
    (2024, 'T3', 'Île-de-France', 'Paris',     'Vêtements',    'Hauts',         70000),
    (2024, 'T3', 'Auvergne-RA',   'Lyon',      'Électronique', 'Smartphones',   38000),
    (2024, 'T3', 'PACA',          'Marseille',  'Vêtements',    'Chaussures',    22000),
    (2024, 'T4', 'Île-de-France', 'Paris',     'Électronique', 'Smartphones', 180000),
    (2024, 'T4', 'Île-de-France', 'Paris',     'Vêtements',    'Hauts',        120000),
    (2024, 'T4', 'Auvergne-RA',   'Lyon',      'Électronique', 'Smartphones',   65000),
    (2024, 'T4', 'PACA',          'Marseille',  'Électronique', 'Laptops',       58000),
    (2023, 'T1', 'Île-de-France', 'Paris',     'Électronique', 'Smartphones', 100000),
    (2023, 'T1', 'Auvergne-RA',   'Lyon',      'Électronique', 'Smartphones',   40000),
    (2023, 'T2', 'Île-de-France', 'Paris',     'Électronique', 'Smartphones', 130000),
    (2023, 'T2', 'Auvergne-RA',   'Lyon',      'Vêtements',    'Hauts',         32000)
) AS t(annee, trimestre, region, ville, categorie, sous_categorie, ca)
""")

print("Table fact_ventes créée !")
sql("SELECT * FROM fact_ventes LIMIT 10")

Vérifions les données disponibles :

In [ ]:
print("=== Résumé des données ===")
print(f"Nombre de lignes : {con.sql('SELECT COUNT(*) FROM fact_ventes').fetchone()[0]}")
print(f"\nAnnées : {con.sql('SELECT DISTINCT annee FROM fact_ventes ORDER BY annee').fetchall()}")
print(f"Trimestres : {con.sql('SELECT DISTINCT trimestre FROM fact_ventes ORDER BY trimestre').fetchall()}")
print(f"Régions : {con.sql('SELECT DISTINCT region FROM fact_ventes ORDER BY region').fetchall()}")
print(f"Catégories : {con.sql('SELECT DISTINCT categorie FROM fact_ventes ORDER BY categorie').fetchall()}")
print(f"\nCA total : {con.sql('SELECT SUM(ca) FROM fact_ventes').fetchone()[0]:,.0f} €")

---
## 2. Opérations OLAP de base

### 2.1 SLICE — Fixer une dimension

**Slice** = on "tranche" le cube en fixant **une dimension** à une valeur.

Exemple : "Je veux voir les ventes, mais **uniquement pour l'Électronique**."

In [ ]:
# SLICE : on fixe la catégorie à 'Électronique'
sql("""
SELECT
    trimestre,
    region,
    SUM(ca) as chiffre_affaires
FROM fact_ventes
WHERE categorie = 'Électronique'     -- ← SLICE : on fixe cette dimension
  AND annee = 2024
GROUP BY trimestre, region
ORDER BY trimestre, region
""")

**Ce qui s'est passé :** on a "tranché" le cube pour ne garder qu'une couche (Électronique). On voit maintenant les ventes par trimestre et région, mais uniquement pour cette catégorie.

### 2.2 DICE — Filtrer sur plusieurs dimensions

**Dice** = on découpe un **sous-cube** en filtrant sur plusieurs dimensions simultanément.

In [ ]:
# DICE : on filtre sur 3 dimensions
sql("""
SELECT
    trimestre,
    categorie,
    SUM(ca) as chiffre_affaires
FROM fact_ventes
WHERE annee = 2024                          -- ← filtre dimension Temps
  AND categorie = 'Électronique'            -- ← filtre dimension Produit
  AND trimestre IN ('T1', 'T2')             -- ← filtre dimension Temps (encore)
  AND region = 'Île-de-France'              -- ← filtre dimension Géographie
GROUP BY trimestre, categorie
ORDER BY trimestre
""")

**Slice vs Dice :**
- Slice = filtre sur **1** dimension (WHERE categorie = ...)
- Dice = filtre sur **2+** dimensions (WHERE categorie = ... AND region = ... AND trimestre IN (...))

### 2.3 ROLL-UP — Agréger (monter dans la hiérarchie)

**Roll-up** = on "zoome arrière" pour voir les données à un niveau plus agrégé.

Hiérarchie géographique : **Ville → Région → France**

In [ ]:
# Niveau le plus détaillé : CA par VILLE
print("=== Niveau VILLE (le plus détaillé) ===")
display(sql("""
SELECT ville, SUM(ca) as ca
FROM fact_ventes WHERE annee = 2024
GROUP BY ville ORDER BY ca DESC
"""))

# Roll-up : CA par RÉGION
print("\n=== Roll-up → Niveau RÉGION ===")
display(sql("""
SELECT region, SUM(ca) as ca
FROM fact_ventes WHERE annee = 2024
GROUP BY region ORDER BY ca DESC
"""))

# Roll-up : CA TOTAL (France entière)
print("\n=== Roll-up → Niveau TOTAL ===")
display(sql("""
SELECT 'France' as pays, SUM(ca) as ca
FROM fact_ventes WHERE annee = 2024
"""))

### 2.4 DRILL-DOWN — Détailler (descendre dans la hiérarchie)

**Drill-down** = l'inverse du roll-up. On "zoome" pour voir plus de détail.

Le CEO regarde le CA annuel → il veut voir par trimestre → puis par mois.

In [ ]:
# Vue agrégée : CA par année
print("=== Le CEO voit le CA annuel ===")
display(sql("""
SELECT annee, SUM(ca) as ca_total
FROM fact_ventes
GROUP BY annee ORDER BY annee
"""))

# Drill-down : "Montre-moi par trimestre pour 2024"
print("\n=== Drill-down → par trimestre ===")
display(sql("""
SELECT annee, trimestre, SUM(ca) as ca_trimestre
FROM fact_ventes
WHERE annee = 2024
GROUP BY annee, trimestre
ORDER BY trimestre
"""))

# Drill-down encore : "Et par catégorie dans chaque trimestre ?"
print("\n=== Drill-down → par trimestre ET catégorie ===")
display(sql("""
SELECT trimestre, categorie, SUM(ca) as ca
FROM fact_ventes
WHERE annee = 2024
GROUP BY trimestre, categorie
ORDER BY trimestre, ca DESC
"""))

### 2.5 PIVOT — Changer de perspective

**Pivot** = on intervertit les lignes et les colonnes pour changer l'angle de vue.

Au lieu de voir les régions en lignes et catégories en colonnes, on fait l'inverse.

In [ ]:
# Avant pivot : régions en lignes
print("=== Avant pivot (régions en lignes) ===")
display(sql("""
SELECT
    region,
    SUM(CASE WHEN categorie = 'Électronique' THEN ca ELSE 0 END) as electronique,
    SUM(CASE WHEN categorie = 'Vêtements' THEN ca ELSE 0 END) as vetements,
    SUM(ca) as total
FROM fact_ventes WHERE annee = 2024
GROUP BY region
ORDER BY total DESC
"""))

# Après pivot : catégories en lignes, régions en colonnes
print("\n=== Après pivot (catégories en lignes, régions en colonnes) ===")
display(sql("""
SELECT
    categorie,
    SUM(CASE WHEN region = 'Île-de-France' THEN ca ELSE 0 END) as "IDF",
    SUM(CASE WHEN region = 'Auvergne-RA' THEN ca ELSE 0 END) as "Auvergne-RA",
    SUM(CASE WHEN region = 'PACA' THEN ca ELSE 0 END) as "PACA",
    SUM(ca) as total
FROM fact_ventes WHERE annee = 2024
GROUP BY categorie
ORDER BY total DESC
"""))

**La technique SQL :** on utilise `SUM(CASE WHEN ... THEN ca ELSE 0 END)` pour transformer les valeurs d'une colonne en colonnes séparées. C'est le pattern "pivot" en SQL standard.

---
## 3. ROLLUP — Sous-totaux hiérarchiques

`GROUP BY ROLLUP(A, B)` produit :
- Les lignes détaillées (A, B)
- Les sous-totaux par A
- Le total général

C'est un **roll-up automatique** intégré dans SQL !

In [ ]:
# ROLLUP : sous-totaux par région, puis total général
sql("""
SELECT
    COALESCE(region, '*** TOTAL ***') as region,
    COALESCE(categorie, '** sous-total **') as categorie,
    SUM(ca) as chiffre_affaires
FROM fact_ventes
WHERE annee = 2024
GROUP BY ROLLUP(region, categorie)
ORDER BY region NULLS LAST, categorie NULLS LAST
""")

**Lecture du résultat :**
- Les lignes avec une catégorie précise = données détaillées
- Les lignes `** sous-total **` = sous-total pour la région
- La ligne `*** TOTAL ***` = total général

`ROLLUP(region, categorie)` produit 3 niveaux d'agrégation :
1. `(region, categorie)` — détail
2. `(region)` — sous-total par région
3. `()` — total général

### ✏️ Exercice : ROLLUP temporel

Écrivez un ROLLUP sur `(annee, trimestre)` pour voir le CA avec sous-totaux par année et total général.

In [ ]:
# ✏️ À VOUS : ROLLUP temporel
# Remplacez les ___ par le bon code

sql("""
SELECT
    COALESCE(CAST(annee AS VARCHAR), 'TOTAL') as annee,
    COALESCE(trimestre, 'sous-total') as trimestre,
    SUM(ca) as chiffre_affaires
FROM fact_ventes
GROUP BY ROLLUP(___, ___)
ORDER BY annee NULLS LAST, trimestre NULLS LAST
""")

---
## 4. CUBE — Toutes les combinaisons

`GROUP BY CUBE(A, B)` produit **toutes les combinaisons possibles** :
- (A, B) — détail
- (A) — sous-total par A
- (B) — sous-total par B ← **la différence avec ROLLUP !**
- () — total général

ROLLUP = hiérarchique (A puis B). CUBE = croisé (A et B dans tous les sens).

In [ ]:
# CUBE : toutes les combinaisons région x catégorie
sql("""
SELECT
    COALESCE(region, '** TOUTES **') as region,
    COALESCE(categorie, '** TOUTES **') as categorie,
    SUM(ca) as chiffre_affaires
FROM fact_ventes
WHERE annee = 2024
GROUP BY CUBE(region, categorie)
ORDER BY region NULLS LAST, categorie NULLS LAST
""")

**Comparez avec le ROLLUP :**

| ROLLUP(region, categorie) | CUBE(region, categorie) |
|---------------------------|-------------------------|
| (region, categorie) ✅ | (region, categorie) ✅ |
| (region) ✅ | (region) ✅ |
| () ✅ | (categorie) ✅ ← **en plus !** |
| | () ✅ |

CUBE ajoute les sous-totaux par catégorie (toutes régions confondues). C'est très utile pour les tableaux croisés.

---
## 5. GROUPING SETS — Combinaisons à la carte

`GROUPING SETS` vous laisse choisir **exactement** quels regroupements vous voulez.

In [ ]:
# GROUPING SETS : exactement ce qu'on veut
sql("""
SELECT
    region,
    categorie,
    trimestre,
    SUM(ca) as chiffre_affaires
FROM fact_ventes
WHERE annee = 2024
GROUP BY GROUPING SETS (
    (region, categorie),     -- CA par région et catégorie
    (trimestre),             -- CA par trimestre (toutes régions, toutes catégories)
    ()                       -- Total général
)
ORDER BY region NULLS LAST, categorie NULLS LAST, trimestre NULLS LAST
""")

**Quand utiliser quoi ?**

| Fonction | Produit | Cas d'usage |
|----------|---------|-------------|
| `ROLLUP(A, B)` | A+B, A, total | Sous-totaux hiérarchiques (région → pays) |
| `CUBE(A, B)` | A+B, A, B, total | Tableau croisé complet |
| `GROUPING SETS(...)` | Exactement ce que vous spécifiez | Rapport personnalisé |

---
## 6. Window Functions — Analyses sans réduire les lignes

Les fonctions de fenêtrage calculent une valeur **sans supprimer de lignes** (contrairement à GROUP BY).

### 6.1 RANK — Classement

In [ ]:
# Classement des régions par CA (par trimestre)
sql("""
SELECT
    trimestre,
    region,
    SUM(ca) as ca,
    RANK() OVER (
        PARTITION BY trimestre      -- classement DANS chaque trimestre
        ORDER BY SUM(ca) DESC       -- du plus gros CA au plus petit
    ) as rang
FROM fact_ventes
WHERE annee = 2024
GROUP BY trimestre, region
ORDER BY trimestre, rang
""")

**Explication :**
- `PARTITION BY trimestre` = on fait un classement **séparé** pour chaque trimestre
- `ORDER BY SUM(ca) DESC` = on classe par CA décroissant
- Le résultat garde **toutes les lignes** (pas de réduction comme GROUP BY)

### 6.2 LAG — Comparaison avec la période précédente

In [ ]:
# Évolution du CA trimestre par trimestre
sql("""
SELECT
    annee,
    trimestre,
    SUM(ca) as ca_trimestre,
    LAG(SUM(ca)) OVER (ORDER BY annee, trimestre) as ca_trimestre_precedent,
    ROUND(
        (SUM(ca) - LAG(SUM(ca)) OVER (ORDER BY annee, trimestre))
        * 100.0 / LAG(SUM(ca)) OVER (ORDER BY annee, trimestre),
        1
    ) as croissance_pct
FROM fact_ventes
GROUP BY annee, trimestre
ORDER BY annee, trimestre
""")

**LAG(valeur) OVER (ORDER BY ...)** = récupère la valeur de la **ligne précédente** dans l'ordre spécifié.

C'est indispensable pour les analyses "vs période précédente" que demandent tous les directeurs.

### 6.3 SUM() OVER — Cumul progressif

In [ ]:
# CA cumulé trimestre par trimestre en 2024
sql("""
SELECT
    trimestre,
    SUM(ca) as ca_trimestre,
    SUM(SUM(ca)) OVER (
        ORDER BY trimestre
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) as ca_cumule
FROM fact_ventes
WHERE annee = 2024
GROUP BY trimestre
ORDER BY trimestre
""")

**SUM() OVER (ORDER BY ... ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)** = cumul depuis le début.

Le CA cumulé permet de voir la progression vers un objectif annuel.

### 6.4 Part du total — Pourcentage

In [ ]:
# Part de chaque région dans le CA total
sql("""
SELECT
    region,
    SUM(ca) as ca_region,
    SUM(SUM(ca)) OVER () as ca_total,
    ROUND(
        SUM(ca) * 100.0 / SUM(SUM(ca)) OVER (),
        1
    ) as part_pct
FROM fact_ventes
WHERE annee = 2024
GROUP BY region
ORDER BY ca_region DESC
""")

**SUM() OVER ()** (sans PARTITION BY ni ORDER BY) = le total de TOUTES les lignes. C'est le dénominateur du pourcentage.

---
## 7. Récapitulatif visuel

| Opération | SQL | Ce que ça fait |
|-----------|-----|----------------|
| **Slice** | `WHERE categorie = 'X'` | Fixe 1 dimension |
| **Dice** | `WHERE cat = 'X' AND region = 'Y'` | Filtre N dimensions |
| **Roll-up** | `GROUP BY region` au lieu de `GROUP BY ville` | Monte dans la hiérarchie |
| **Drill-down** | `GROUP BY annee, trimestre, mois` | Descend dans la hiérarchie |
| **Pivot** | `SUM(CASE WHEN region = 'X' THEN ca END)` | Lignes ↔ Colonnes |
| **ROLLUP** | `GROUP BY ROLLUP(A, B)` | Sous-totaux hiérarchiques |
| **CUBE** | `GROUP BY CUBE(A, B)` | Toutes les combinaisons |
| **GROUPING SETS** | `GROUP BY GROUPING SETS((A,B), (A), ())` | Combinaisons à la carte |
| **RANK** | `RANK() OVER (ORDER BY ca DESC)` | Classement |
| **LAG** | `LAG(ca) OVER (ORDER BY mois)` | Valeur précédente |
| **Cumul** | `SUM(ca) OVER (ORDER BY mois ROWS ...)` | Total cumulé |
| **% total** | `ca / SUM(ca) OVER ()` | Part du total |

---

## ✏️ Exercices à faire vous-même

### Exercice 1 : ROLLUP sur 3 niveaux
Écrivez un ROLLUP sur `(region, ville, categorie)` pour 2024.

In [ ]:
# ✏️ Exercice 1 : à vous !
sql("""
-- Écrivez votre requête ROLLUP ici
SELECT 'Remplacez cette requête' as message
""")

### Exercice 2 : Rapport de direction

Écrivez UNE SEULE requête qui affiche pour chaque trimestre 2024 :
- Le CA par catégorie
- Le rang de chaque catégorie dans le trimestre
- La part (%) de chaque catégorie dans le CA du trimestre
- Le CA du trimestre précédent pour la même catégorie

In [ ]:
# ✏️ Exercice 2 : rapport de direction
sql("""
-- Combinez GROUP BY, RANK(), SUM() OVER (PARTITION BY...), et LAG()
-- Indice : commencez par le GROUP BY, puis ajoutez les window functions une par une
SELECT 'Remplacez cette requête' as message
""")

### Exercice 3 : GROUPING SETS personnalisé

Créez un rapport qui montre dans une seule requête :
- Le CA par catégorie (toutes régions confondues)
- Le CA par région (toutes catégories confondues)
- Le CA total

In [ ]:
# ✏️ Exercice 3 : GROUPING SETS
sql("""
-- Utilisez GROUPING SETS avec 3 sets : (categorie), (region), ()
SELECT 'Remplacez cette requête' as message
""")